In [10]:
import os
import re
import csv
import pandas as pd
from numpy import trapz


def parse_graph_stats(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    name_match = re.search(r'графе (.+?):', content)
    graph_name = name_match.group(1).strip() if name_match else os.path.basename(filepath)

    # Степени узлов
    try:
        min_deg = float(re.search(r'Минимальная степень узлов: (\d+)', content).group(1))
        avg_deg = float(re.search(r'Средняя степень узлов: ([\d.]+)', content).group(1))
        max_deg = float(re.search(r'Максимальная степень узлов: (\d+)', content).group(1))
    except:
        min_deg, avg_deg, max_deg = None, None, None

    # Удаление x% узлов
    def extract_block(header):
        match = re.search(header + r':\s*((?:\n\t+x = [^\n]+)+)', content)
        return match.group(1) if match else ''

    def extract_removal_data(block):
        data = {}
        for match in re.findall(r'x = ([\d.]+)%: доля вершин в наибольшей WCC: ([\d.]+)', block):
            x = float(match[0])
            y = float(match[1])
            data[x] = y
        return data

    block_random = extract_block(r'Удаление случайных узлов')
    block_degree = extract_block(r'Удаление узлов наибольшей степени')

    data_random = extract_removal_data(block_random)
    data_degree = extract_removal_data(block_degree)

    def compute_auc(data):
        if not data: return None
        x = sorted(data.keys())
        y = [data[k] for k in x]
        return trapz(y, x)

    auc_random = compute_auc(data_random)
    auc_degree = compute_auc(data_degree)

    return {
        'graph': graph_name,
        'min_deg': min_deg,
        'avg_deg': avg_deg,
        'max_deg': max_deg,
        'auc_random': auc_random,
        'auc_degree': auc_degree
    }



input_dir = r'C:\Users\Dmitry\Desktop\Graph_Theory_LeetCode25\src\output'
output_csv = os.path.join(r'C:\Users\Dmitry\Desktop\Graph_Theory_LeetCode25\src\visualization', 'graph_resilience.csv')

results = []

for file in os.listdir(input_dir):
    if file.endswith('.txt'):
        full_path = os.path.join(input_dir, file)
        stats = parse_graph_stats(full_path)
        results.append(stats)


with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=[
        'graph', 'min_deg', 'avg_deg', 'max_deg', 'auc_random', 'auc_degree'
    ])
    writer.writeheader()
    for row in results:
        writer.writerow(row)

print("CSV сохранён:", output_csv)

df = pd.read_csv(output_csv)
display(df)


CSV сохранён: C:\Users\Dmitry\Desktop\Graph_Theory_LeetCode25\src\visualization\graph_resilience.csv


C:\Users\Dmitry\AppData\Local\Temp\ipykernel_15556\3677929602.py:46: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return trapz(y, x)


,graph,min_deg,avg_deg,max_deg,auc_random,auc_degree
0,CA-AstroPh,0.0,21.10,504.0,64.16075,21.497750
1,ca-coauthors-dblp,1.0,56.41,3299.0,71.48700,46.063000
2,CA-GrQc,0.0,5.53,81.0,35.76425,2.038000
3,com-orkut.ungraph,1.0,76.28,33313.0,77.68450,54.879250
4,com-youtube.ungraph,1.0,5.27,28754.0,45.92450,3.143976
5,Email-EuAll,0.0,2.75,7636.0,7.14980,0.090664
6,musae_git_edges,1.0,15.33,9458.0,66.19150,9.118000
7,soc-wiki-Vote,1.0,6.56,102.0,55.87100,11.151250
8,vk,1.0,10.83,6503.0,60.08175,12.474500
9,web-Google,1.0,9.87,6332.0,53.05725,5.091250
